# 02 Preprocess Data
This notebook will preprocess the downloaded datasets from `01_fetch_data.ipynb` and output structured parquets.

Each dataset (`_yellow`, `_green`, `_fhvhv`, `_ridership`, `_alerts`, `_weather`) will undergo the following pipe:
1. `sdf_dataset`: initial read of .parquet or .csv as a pyspark parquet
2. `sdf_dataset_clean`: parquet after removing rows with data anomalies
3. `sdf_dataset_selected`: parquet after removing columns unused by downstream analysis

Note, `_fhvhv` is large, so it is saved with a monthly for-loop. All other datasets are saved in one go.

In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "America/New_York") # timezone set to NYC
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR") # silence warnings

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/26 23:20:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from functools import reduce
from operator import and_
from pathlib import Path
from pyspark.sql import functions as F

# output directory
processed_output_dir = "../data/processed"

# specify time range of data
YEARS = [2025, 2026]
MONTHS = range(1, 13)

# Analysis time window: Jan 2025 to May 2026 inclusive
START_TIME = "2025-01-01 00:00:00"
END_TIME = "2026-06-01 00:00:00"   # exclusive

In [3]:
from pyspark import StorageLevel

cleaning_log = []

def apply_cleaning_step(sdf, step_name, condition, n_before):
    """
    Function to applying filter step and print no. rows before vs. after for report
    """
    sdf_after = sdf.filter(condition)
    n_after = sdf_after.count()

    removed = n_before - n_after
    pct = removed / n_before if n_before else 0
    pct_raw = removed / n_raw if n_raw else 0

    cleaning_log.append((step_name, n_before, n_after, removed, pct))

    print(f"{step_name}")
    print(f"Before:  {n_before:,}")
    print(f"After:   {n_after:,}")
    print(f"Removed: {removed:,} ({pct_raw:.2%})\n")

    return sdf_after, n_after

## Preprocess TLC Trip Record Data (Yellow)
We want to retain all rows with data anomaly, such as the following: 
- trips with `trip_distance` <= 0
- trips with `total_amount` == 0
- trips with `RatecodeID` == 99 (Null/unknown)
- trips with `payment_type` == 6 (Voided trip)
- trips with `tpep_dropoff_datetime` - `tpep_pickup_datetime` duration < 60 seconds or > 8 hours
- trips with < 0 for any money related columns
- trips with NULL in any of `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `trip_distance`, `total_amount`

In [14]:
# Read the parquet from directory (all months combined)
sdf_yellow = spark.read.parquet('../data/raw/tlc_trips/yellow')

# Print schema for reference
sdf_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [15]:
# Start with un-filtered data and raw count
n = sdf_yellow.count()

# Remove rows with data anomalies
required_money_cols = ["fare_amount", "mta_tax", "extra", "improvement_surcharge", "tip_amount", "tolls_amount", "total_amount"]
nullable_money_cols = ["congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]

sdf_yellow_clean = (
    sdf_yellow
    .withColumn( # calculate trip duration
        "trip_time",
        F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
    )
)

print(f"Raw rows: {n:,}\n")

Raw rows: 67,721,884



In [16]:
# Step 1: valid trip fields
valid_trip = (
    (F.col("passenger_count").between(1, 5)) &                                         # passenger_count must be valid
    (F.col("trip_distance") > 0) &                                                     # trip_distance must be positive
    (F.col("total_amount") > 0) &                                                      # total_amount must be positive
    (F.col("RatecodeID").between(1, 6)) &                                              # RatecodeID must not be Null/unknown
    (F.col("payment_type").between(1, 5))                                              # payment_type must not be Voided trip
)
sdf_yellow_clean, n = apply_cleaning_step(sdf_yellow_clean, "Valid trip fields", valid_trip, n)

# Step 2: reasonable trip time
reasonable_time = F.col("trip_time").between(60, 8 * 60 * 60)                          # duration must be between 60 seconds to 8 hours
sdf_yellow_clean, n = apply_cleaning_step(sdf_yellow_clean, "Reasonable trip time", reasonable_time, n)

# Step 3: required pickup/dropoff information
required_trip_info = (
    (F.col("tpep_pickup_datetime") >= F.lit(START_TIME)) &                             # trip must begin within validated time range
    (F.col("tpep_pickup_datetime") < F.lit(END_TIME)) &                                # trip must begin within validated time range
    F.col("tpep_pickup_datetime").isNotNull() &                                        # tpep_pickup_datetime must not be NULL
    F.col("tpep_dropoff_datetime").isNotNull() &                                       # tpep_dropoff_datetime must not be NULL
    F.col("PULocationID").isNotNull() &                                                # PULocationID must not be Null
    F.col("DOLocationID").isNotNull()                                                  # DOLocationID must not be Null
)
sdf_yellow_clean, n = apply_cleaning_step(sdf_yellow_clean, "Required pickup/dropoff information", required_trip_info, n)

# Step 4: valid money columns
valid_money = (
    reduce(and_, [F.col(c) >= 0 for c in required_money_cols]) &                       # These columns must be non-negative and not NULL
    reduce(and_, [F.col(c).isNull() | (F.col(c) >= 0) for c in nullable_money_cols])   # These columns must be non-negative, can be NULL
)
sdf_yellow_clean, n = apply_cleaning_step(sdf_yellow_clean, "Valid money columns", valid_money, n)

Valid trip fields
Before:  67,721,884
After:   47,909,254
Removed: 19,812,630 (29.26%)



Reasonable trip time
Before:  47,909,254
After:   46,955,474
Removed: 953,780 (1.99%)



Required pickup/dropoff information
Before:  46,955,474
After:   46,955,443
Removed: 31 (0.00%)



Valid money columns
Before:  46,955,443
After:   46,955,438
Removed: 5 (0.00%)



In [ ]:
# Remove columns unused in downstream analysis
sdf_yellow_selected = (
    sdf_yellow_clean
    .select(
        F.col("tpep_pickup_datetime").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        (F.col("total_amount") - F.col("tip_amount")).cast("decimal(12,2)").alias("subtotal"), # passenger payment total excluding tips, to 2 d.p.
        F.lit("yellow").alias("service_type"),
    )
)

sdf_yellow_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/tlc_trips/yellow")

Specifying dataset shape:
- `sdf_yellow`, raw parquet file, contains 67721884 rows x 20 columns
- `sdf_yellow_clean`, filtered for rows with valid data, contains 46,955,438 rows x 20 columns 
- `sdf_yellow_selected`, filtered for columns to be used downstream, contains 46,955,438 rows x 6 columns

---

## Preprocess TLC Trip Record Data (Green)
We want to remove all rows with data anomaly, such as the following: 
- trips with `trip_distance` <= 0
- trips with `total_amount` == 0
- trips with `RatecodeID` == 99 (Null/unknown)
- trips with `payment_type` == 6 (Voided trip)
- trips with `lpep_dropoff_datetime` - `lpep_pickup_datetime` duration < 60 seconds or > 8 hours
- trips with < 0 for any money related columns
- trips with NULL in any of `lpep_pickup_datetime`, `lpep_dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `trip_distance`, `total_amount`

Further, since green taxi trips include street-hail (`trip_type`==1) and dispatch (`trip_type`==2), we remove where `trip_type`==2 so the green taxi cohort can be combined with the yellow taxi cohort as street-hail services, in contrast to the dispath only services provided in FHVHV.

In [22]:
# Read the parquet from directory (all months combined)
sdf_green = spark.read.parquet('../data/raw/tlc_trips/green')

# Print schema for reference
sdf_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [23]:
# Start with un-filtered data and raw count
n = sdf_green.count()

# Remove rows with data anomalies
required_money_cols = ["fare_amount", "mta_tax", "extra", "improvement_surcharge", "tip_amount", "tolls_amount", "total_amount"]
nullable_money_cols = ["congestion_surcharge", "cbd_congestion_fee"]

sdf_green_clean = (
    sdf_green
    .withColumn( # calculate trip duration
        "trip_time",
        F.unix_timestamp("lpep_dropoff_datetime") - F.unix_timestamp("lpep_pickup_datetime")
    )
)

print(f"Raw rows: {n:,}\n")

# Step 1: valid trip fields
valid_trip = (
    (F.col("trip_type") == 1) &                                                        # trip_type must be street-hail
    (F.col("passenger_count").between(1, 5)) &                                         # passenger_count must be valid
    (F.col("trip_distance") > 0) &                                                     # trip_distance must be positive
    (F.col("total_amount") > 0) &                                                      # total_amount must be positive
    (F.col("RatecodeID").between(1, 6)) &                                              # RatecodeID must not be Null/unknown
    (F.col("payment_type").between(1, 5))                                              # payment_type must not be Voided trip
)
sdf_green_clean, n = apply_cleaning_step(sdf_green_clean, "Valid trip fields", valid_trip, n)

# Step 2: reasonable trip time
reasonable_time = F.col("trip_time").between(60, 8 * 60 * 60)                          # duration must be between 60 seconds to 8 hours
sdf_green_clean, n = apply_cleaning_step(sdf_green_clean, "Reasonable trip time", reasonable_time, n)

# Step 3: required pickup/dropoff information
required_trip_info = (
    (F.col("lpep_pickup_datetime") >= F.lit(START_TIME)) &                             # trip must begin within validated time range
    (F.col("lpep_pickup_datetime") < F.lit(END_TIME)) &                                # trip must begin within validated time range
    F.col("lpep_pickup_datetime").isNotNull() &                                        # lpep_pickup_datetime must not be NULL
    F.col("lpep_dropoff_datetime").isNotNull() &                                       # lpep_dropoff_datetime must not be NULL
    F.col("PULocationID").isNotNull() &                                                # PULocationID must not be Null
    F.col("DOLocationID").isNotNull()                                                  # DOLocationID must not be Null
)
sdf_green_clean, n = apply_cleaning_step(sdf_green_clean, "Required pickup/dropoff information", required_trip_info, n)

# Step 4: valid money columns
valid_money = (
    reduce(and_, [F.col(c) >= 0 for c in required_money_cols]) &                       # These columns must be non-negative and not NULL
    reduce(and_, [F.col(c).isNull() | (F.col(c) >= 0) for c in nullable_money_cols])   # These columns must be non-negative, can be NULL
)
sdf_green_clean, n = apply_cleaning_step(sdf_green_clean, "Valid money columns", valid_money, n)

Raw rows: 802,387

Valid trip fields
Before:  802,387
After:   643,326
Removed: 159,061 (0.23%)

Reasonable trip time
Before:  643,326
After:   637,287
Removed: 6,039 (0.01%)

Required pickup/dropoff information
Before:  637,287
After:   637,282
Removed: 5 (0.00%)

Valid money columns
Before:  637,282
After:   637,282
Removed: 0 (0.00%)



In [8]:
# Remove columns unused in downstream analysis
sdf_green_selected = (
    sdf_green_clean
    .select(
        F.col("lpep_pickup_datetime").alias("pickup_datetime"),
        F.col("lpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        (F.col("total_amount") - F.col("tip_amount")).cast("decimal(12,2)").alias("subtotal"),  # passenger payment total excluding tips, to 2 d.p.
        F.lit("green").alias("service_type"),
    )
)

sdf_green_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/tlc_trips/green")

Specifying dataset shape:
- `sdf_green`, raw parquet file, contains 802387 rows x 20 columns
- `sdf_green_clean`, filtered for rows with valid data, contains 637,282 rows x 20 columns 
- `sdf_green_selected`, filtered for columns to be used downstream, contains 637,282 rows x 6 columns

---

## Preprocess TLC Trip Record Data (FHVHV)
We want to remove all rows with data anomaly, such as the following: 
- trips with `trip_miles` <= 0
- trips with `total_amount` == 0
- trips with `RatecodeID` == 99 (Null/unknown)
- trips with `payment_type` == 6 (Voided trip)
- trips with `trip_time` duration < 60 seconds or > 8 hours
- trips with < 0 for any money related columns
- trips with NULL in any of `pickup_datetime`, `dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `trip_distance`, `total_amount`

In [14]:
# Read the parquet from directory (all months combined)
sdf_fhvhv = spark.read.parquet('../data/raw/tlc_trips/fhvhv')

# Print schema for reference
sdf_fhvhv.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [ ]:
# Chunk the fhvhv data into half-years to avoid spark crash
fhvhv_month_chunks = {
    1: [(2025, m) for m in range(1, 7)],   # Jan-Jun 2025
    2: [(2025, m) for m in range(7, 13)],  # Jul-Dec 2025
    3: [(2026, m) for m in range(1, 6)],   # Jan-May 2026
}

In [ ]:
def process_fhvhv_months(months_to_process):
    """ 
    Process FHVHV months (batch at a time) with the appropriate preprocessing steps
    """
    raw_fhvhv_dir = Path("../data/raw/tlc_trips/fhvhv")
    processed_fhvhv_dir = Path("../data/processed/tlc_trips/fhvhv")

    chunk_log = {
        "Raw rows": 0,
        "Required pickup/dropoff information": [0, 0],
        "Valid request/pickup/dropoff sequence": [0, 0],
        "Reasonable wait time": [0, 0],
        "Valid trip fields": [0, 0],
        "Valid money columns": [0, 0],
    }

    def apply_fhvhv_step(sdf, step_name, condition, n_before):
        """ 
        Keep log of pre and after filter dtaframe sizes
        """
        sdf_after = sdf.filter(condition)
        n_after = sdf_after.count()

        chunk_log[step_name][0] += n_before
        chunk_log[step_name][1] += n_after

        return sdf_after, n_after

    # Process per month in batch
    for year, month in months_to_process:
        input_path = raw_fhvhv_dir / f"{year}-{month:02d}.parquet"
        output_path = processed_fhvhv_dir / f"{year}-{month:02d}.parquet"

        if (output_path / "_SUCCESS").exists():
            print(f"Skipping FHVHV {year}-{month:02d}")
            continue

        print(f"Processing FHVHV {year}-{month:02d}")

        sdf_fhvhv_month = spark.read.parquet(str(input_path))
        n = sdf_fhvhv_month.count()
        chunk_log["Raw rows"] += n

        required_money_cols = ["base_passenger_fare", "tolls", "bcf", "sales_tax", "tips", "driver_pay"]
        nullable_money_cols = ["congestion_surcharge", "airport_fee", "cbd_congestion_fee"]

        sdf_fhvhv_processed = sdf_fhvhv_month

        # Step 1: required pickup/dropoff information
        required_trip_info = (
            F.col("pickup_datetime").isNotNull() &                                             # pickup_datetime must not be Null
            F.col("dropoff_datetime").isNotNull() &                                            # dropoff_datetime must not be Null
            F.col("request_datetime").isNotNull() &                                            # request_datetime must not be Null
            F.col("PULocationID").isNotNull() &                                                # PULocationID must not be Null
            F.col("DOLocationID").isNotNull() &                                                # DOLocationID must not be Null
            (F.col("request_datetime") >= F.lit(START_TIME)) &                                 # request time must be within specified time range
            (F.col("request_datetime") < F.lit(END_TIME))                                      # request time must be within specified time range
        )
        sdf_fhvhv_processed, n = apply_fhvhv_step(
            sdf_fhvhv_processed,
            "Required pickup/dropoff information",
            required_trip_info,
            n
        )

        # Step 2: valid request/pickup/dropoff sequence
        valid_time_sequence = (
            (F.col("pickup_datetime") >= F.col("request_datetime")) &                          # request must be before pickup
            (F.col("dropoff_datetime") > F.col("pickup_datetime"))                             # pickup_datetime must be before dropoff_datetime
        )
        sdf_fhvhv_processed, n = apply_fhvhv_step(
            sdf_fhvhv_processed,
            "Valid request/pickup/dropoff sequence",
            valid_time_sequence,
            n
        )

        # Step 3: reasonable wait time
        reasonable_wait_time = (
            F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
        ).between(0, 60 * 60)                                                                  # wait time must be between 0 seconds and 1 hour
        sdf_fhvhv_processed, n = apply_fhvhv_step(
            sdf_fhvhv_processed,
            "Reasonable wait time",
            reasonable_wait_time,
            n
        )

        # Step 4: valid trip fields
        valid_trip = (
            (F.col("trip_miles") > 0) &                                                        # trip_miles must be positive
            (F.col("trip_time").between(60, 8 * 60 * 60))                                      # trip_time must be between 60 seconds to 8 hours
        )
        sdf_fhvhv_processed, n = apply_fhvhv_step(
            sdf_fhvhv_processed,
            "Valid trip fields",
            valid_trip,
            n
        )

        # Step 5: valid money columns
        valid_money = (
            reduce(and_, [F.col(c) >= 0 for c in required_money_cols]) &                       # These columns must be non-negative and not NULL
            reduce(and_, [F.col(c).isNull() | (F.col(c) >= 0) for c in nullable_money_cols])   # These columns must be non-negative, can be Null
        )
        sdf_fhvhv_processed, n = apply_fhvhv_step(
            sdf_fhvhv_processed,
            "Valid money columns",
            valid_money,
            n
        )

        # Remove columns unused in downstream analysis
        sdf_fhvhv_selected = (
            sdf_fhvhv_processed
            .select(
                F.col("request_datetime"),
                F.col("pickup_datetime"),
                F.col("dropoff_datetime"),
                F.col("PULocationID").cast("int"),
                F.col("DOLocationID").cast("int"),
                (
                    F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
                ).cast("long").alias("wait_time"),                                             # request-to-pickup wait time in seconds
                (
                    F.coalesce(F.col("base_passenger_fare"), F.lit(0))
                    + F.coalesce(F.col("tolls"), F.lit(0))
                    + F.coalesce(F.col("bcf"), F.lit(0))
                    + F.coalesce(F.col("sales_tax"), F.lit(0))
                    + F.coalesce(F.col("congestion_surcharge"), F.lit(0))
                    + F.coalesce(F.col("airport_fee"), F.lit(0))
                    + F.coalesce(F.col("cbd_congestion_fee"), F.lit(0))
                ).cast("decimal(12,2)").alias("subtotal"),                                     # passenger payment total excluding tips
                F.lit("fhvhv").alias("service_type"),
            )
        )

        (
            sdf_fhvhv_selected
            .write
            .mode("overwrite")
            .option("parquet.enable.dictionary", "false")
            .parquet(str(output_path))
        )

        # Clear cache to avoid clogging spark session
        spark.catalog.clearCache()
        del sdf_fhvhv_month
        del sdf_fhvhv_processed
        del sdf_fhvhv_selected

        print(f"\nChunk raw rows: {chunk_log['Raw rows']:,}\n")

        # Print dataframe details
        for step_name, counts in chunk_log.items():
            if step_name == "Raw rows":
                continue

            rows_before, rows_after = counts
            removed = rows_before - rows_after
            pct = removed / rows_before if rows_before else 0

            print(f"{step_name}")
            print(f"Before:  {rows_before:,}")
            print(f"After:   {rows_after:,}")
            print(f"Removed: {removed:,} ({pct:.2%})\n")
            
        return chunk_log

In [ ]:
# Pre-process Jan-Jun 2025
fhvhv_log_1 = process_fhvhv_months(fhvhv_month_chunks[1])

Processing FHVHV 2025-01



Chunk raw rows: 20,405,666

Required pickup/dropoff information
Before:  20,405,666
After:   20,404,502
Removed: 1,164 (0.01%)

Valid request/pickup/dropoff sequence
Before:  20,404,502
After:   20,204,711
Removed: 199,791 (0.98%)

Reasonable wait time
Before:  20,204,711
After:   20,204,006
Removed: 705 (0.00%)

Valid trip fields
Before:  20,204,006
After:   20,200,575
Removed: 3,431 (0.02%)

Valid money columns
Before:  20,200,575
After:   20,197,866
Removed: 2,709 (0.01%)

Processing FHVHV 2025-02



Chunk raw rows: 39,745,127

Required pickup/dropoff information
Before:  39,745,127
After:   39,743,963
Removed: 1,164 (0.00%)

Valid request/pickup/dropoff sequence
Before:  39,743,963
After:   39,350,415
Removed: 393,548 (0.99%)

Reasonable wait time
Before:  39,350,415
After:   39,349,330
Removed: 1,085 (0.00%)

Valid trip fields
Before:  39,349,330
After:   39,342,653
Removed: 6,677 (0.02%)

Valid money columns
Before:  39,342,653
After:   39,339,205
Removed: 3,448 (0.01%)

Processing FHVHV 2025-03



Chunk raw rows: 60,282,006

Required pickup/dropoff information
Before:  60,282,006
After:   60,280,842
Removed: 1,164 (0.00%)

Valid request/pickup/dropoff sequence
Before:  60,280,842
After:   59,669,342
Removed: 611,500 (1.01%)

Reasonable wait time
Before:  59,669,342
After:   59,665,145
Removed: 4,197 (0.01%)

Valid trip fields
Before:  59,665,145
After:   59,655,336
Removed: 9,809 (0.02%)

Valid money columns
Before:  59,655,336
After:   59,651,478
Removed: 3,858 (0.01%)

Processing FHVHV 2025-04



Chunk raw rows: 80,035,989

Required pickup/dropoff information
Before:  80,035,989
After:   80,034,825
Removed: 1,164 (0.00%)

Valid request/pickup/dropoff sequence
Before:  80,034,825
After:   79,217,894
Removed: 816,931 (1.02%)

Reasonable wait time
Before:  79,217,894
After:   79,213,282
Removed: 4,612 (0.01%)

Valid trip fields
Before:  79,213,282
After:   79,200,182
Removed: 13,100 (0.02%)

Valid money columns
Before:  79,200,182
After:   79,195,948
Removed: 4,234 (0.01%)

Processing FHVHV 2025-05



Chunk raw rows: 101,127,182

Required pickup/dropoff information
Before:  101,127,182
After:   101,126,018
Removed: 1,164 (0.00%)

Valid request/pickup/dropoff sequence
Before:  101,126,018
After:   100,073,734
Removed: 1,052,284 (1.04%)

Reasonable wait time
Before:  100,073,734
After:   100,068,550
Removed: 5,184 (0.01%)

Valid trip fields
Before:  100,068,550
After:   100,051,249
Removed: 17,301 (0.02%)

Valid money columns
Before:  100,051,249
After:   100,046,656
Removed: 4,593 (0.00%)

Processing FHVHV 2025-06



Chunk raw rows: 120,995,191

Required pickup/dropoff information
Before:  120,995,191
After:   120,994,027
Removed: 1,164 (0.00%)

Valid request/pickup/dropoff sequence
Before:  120,994,027
After:   119,710,411
Removed: 1,283,616 (1.06%)

Reasonable wait time
Before:  119,710,411
After:   119,704,629
Removed: 5,782 (0.00%)

Valid trip fields
Before:  119,704,629
After:   119,683,580
Removed: 21,049 (0.02%)

Valid money columns
Before:  119,683,580
After:   119,678,487
Removed: 5,093 (0.00%)



In [ ]:
# Pre-process Jul-Dec 2025
fhvhv_log_2 = process_fhvhv_months(fhvhv_month_chunks[2])

Processing FHVHV 2025-07



Chunk raw rows: 19,653,012

Required pickup/dropoff information
Before:  19,653,012
After:   19,653,012
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  19,653,012
After:   19,427,656
Removed: 225,356 (1.15%)

Reasonable wait time
Before:  19,427,656
After:   19,427,244
Removed: 412 (0.00%)

Valid trip fields
Before:  19,427,244
After:   19,423,997
Removed: 3,247 (0.02%)

Valid money columns
Before:  19,423,997
After:   19,423,526
Removed: 471 (0.00%)

Processing FHVHV 2025-08



Chunk raw rows: 38,924,473

Required pickup/dropoff information
Before:  38,924,473
After:   38,924,473
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  38,924,473
After:   38,474,807
Removed: 449,666 (1.16%)

Reasonable wait time
Before:  38,474,807
After:   38,473,937
Removed: 870 (0.00%)

Valid trip fields
Before:  38,473,937
After:   38,467,498
Removed: 6,439 (0.02%)

Valid money columns
Before:  38,467,498
After:   38,466,314
Removed: 1,184 (0.00%)

Processing FHVHV 2025-09



Chunk raw rows: 58,359,114

Required pickup/dropoff information
Before:  58,359,114
After:   58,359,114
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  58,359,114
After:   57,661,309
Removed: 697,805 (1.20%)

Reasonable wait time
Before:  57,661,309
After:   57,660,053
Removed: 1,256 (0.00%)

Valid trip fields
Before:  57,660,053
After:   57,650,406
Removed: 9,647 (0.02%)

Valid money columns
Before:  57,650,406
After:   57,648,232
Removed: 2,174 (0.00%)

Processing FHVHV 2025-10



Chunk raw rows: 79,667,815

Required pickup/dropoff information
Before:  79,667,815
After:   79,667,815
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  79,667,815
After:   78,706,473
Removed: 961,342 (1.21%)

Reasonable wait time
Before:  78,706,473
After:   78,704,904
Removed: 1,569 (0.00%)

Valid trip fields
Before:  78,704,904
After:   78,691,990
Removed: 12,914 (0.02%)

Valid money columns
Before:  78,691,990
After:   78,688,437
Removed: 3,553 (0.00%)

Processing FHVHV 2025-11



Chunk raw rows: 100,486,055

Required pickup/dropoff information
Before:  100,486,055
After:   100,486,055
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  100,486,055
After:   99,256,206
Removed: 1,229,849 (1.22%)

Reasonable wait time
Before:  99,256,206
After:   99,254,186
Removed: 2,020 (0.00%)

Valid trip fields
Before:  99,254,186
After:   99,238,166
Removed: 16,020 (0.02%)

Valid money columns
Before:  99,238,166
After:   99,232,967
Removed: 5,199 (0.01%)

Processing FHVHV 2025-12



Chunk raw rows: 122,594,493

Required pickup/dropoff information
Before:  122,594,493
After:   122,594,493
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  122,594,493
After:   121,075,485
Removed: 1,519,008 (1.24%)

Reasonable wait time
Before:  121,075,485
After:   121,070,991
Removed: 4,494 (0.00%)

Valid trip fields
Before:  121,070,991
After:   121,051,336
Removed: 19,655 (0.02%)

Valid money columns
Before:  121,051,336
After:   121,043,389
Removed: 7,947 (0.01%)



In [ ]:
# Pre-process Jan-May 2026
fhvhv_log_3 = process_fhvhv_months(fhvhv_month_chunks[3])

Processing FHVHV 2026-01



Chunk raw rows: 20,940,373

Required pickup/dropoff information
Before:  20,940,373
After:   20,940,373
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  20,940,373
After:   20,672,615
Removed: 267,758 (1.28%)

Reasonable wait time
Before:  20,672,615
After:   20,670,448
Removed: 2,167 (0.01%)

Valid trip fields
Before:  20,670,448
After:   20,666,568
Removed: 3,880 (0.02%)

Valid money columns
Before:  20,666,568
After:   20,637,189
Removed: 29,379 (0.14%)

Processing FHVHV 2026-02



Chunk raw rows: 40,816,059

Required pickup/dropoff information
Before:  40,816,059
After:   40,816,059
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  40,816,059
After:   40,267,679
Removed: 548,380 (1.34%)

Reasonable wait time
Before:  40,267,679
After:   40,264,290
Removed: 3,389 (0.01%)

Valid trip fields
Before:  40,264,290
After:   40,257,485
Removed: 6,805 (0.02%)

Valid money columns
Before:  40,257,485
After:   40,224,238
Removed: 33,247 (0.08%)

Processing FHVHV 2026-03



Chunk raw rows: 62,874,417

Required pickup/dropoff information
Before:  62,874,417
After:   62,874,417
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  62,874,417
After:   62,031,228
Removed: 843,189 (1.34%)

Reasonable wait time
Before:  62,031,228
After:   62,024,137
Removed: 7,091 (0.01%)

Valid trip fields
Before:  62,024,137
After:   62,013,708
Removed: 10,429 (0.02%)

Valid money columns
Before:  62,013,708
After:   61,974,976
Removed: 38,732 (0.06%)

Processing FHVHV 2026-04



Chunk raw rows: 83,870,370

Required pickup/dropoff information
Before:  83,870,370
After:   83,870,370
Removed: 0 (0.00%)

Valid request/pickup/dropoff sequence
Before:  83,870,370
After:   82,758,223
Removed: 1,112,147 (1.33%)

Reasonable wait time
Before:  82,758,223
After:   82,750,433
Removed: 7,790 (0.01%)

Valid trip fields
Before:  82,750,433
After:   82,737,290
Removed: 13,143 (0.02%)

Valid money columns
Before:  82,737,290
After:   82,693,683
Removed: 43,607 (0.05%)

Processing FHVHV 2026-05



Chunk raw rows: 105,996,114

Required pickup/dropoff information
Before:  105,996,114
After:   105,996,098
Removed: 16 (0.00%)

Valid request/pickup/dropoff sequence
Before:  105,996,098
After:   104,608,431
Removed: 1,387,667 (1.31%)

Reasonable wait time
Before:  104,608,431
After:   104,599,676
Removed: 8,755 (0.01%)

Valid trip fields
Before:  104,599,676
After:   104,583,066
Removed: 16,610 (0.02%)

Valid money columns
Before:  104,583,066
After:   104,532,497
Removed: 50,569 (0.05%)



In [ ]:
fhvhv_logs = [fhvhv_log_1, fhvhv_log_2, fhvhv_log_3]
fhvhv_log_all = {"Raw rows": 0}

for log in fhvhv_logs:
    fhvhv_log_all["Raw rows"] += log["Raw rows"]

    for step_name, counts in log.items():
        if step_name == "Raw rows":
            continue

        if step_name not in fhvhv_log_all:
            fhvhv_log_all[step_name] = [0, 0]

        fhvhv_log_all[step_name][0] += counts[0]
        fhvhv_log_all[step_name][1] += counts[1]

n_raw = fhvhv_log_all["Raw rows"]

print(f"FHVHV raw rows: {n_raw:,}\n")

for step_name, counts in fhvhv_log_all.items():
    if step_name == "Raw rows":
        continue

    rows_before, rows_after = counts
    removed = rows_before - rows_after
    pct_raw = removed / n_raw if n_raw else 0

    print(f"{step_name}")
    print(f"Before:  {rows_before:,}")
    print(f"After:   {rows_after:,}")
    print(f"Removed: {removed:,} ({pct_raw:.2%} of raw rows)\n")

n_clean = list(fhvhv_log_all.values())[-1][1]
total_removed = n_raw - n_clean
pct_total = total_removed / n_raw if n_raw else 0

print(f"FHVHV clean rows: {n_clean:,}")
print(f"Total removed:    {total_removed:,} ({pct_total:.2%} of raw rows)")


FHVHV raw rows: 349,585,798

Required pickup/dropoff information
Before:  349,585,798
After:   349,584,618
Removed: 1,180 (0.00% of raw rows)

Valid request/pickup/dropoff sequence
Before:  349,584,618
After:   345,394,327
Removed: 4,190,291 (1.20% of raw rows)

Reasonable wait time
Before:  345,394,327
After:   345,375,296
Removed: 19,031 (0.01% of raw rows)

Valid trip fields
Before:  345,375,296
After:   345,317,982
Removed: 57,314 (0.02% of raw rows)

Valid money columns
Before:  345,317,982
After:   345,254,373
Removed: 63,609 (0.02% of raw rows)

FHVHV clean rows: 345,254,373
Total removed:    4,331,425 (1.24% of raw rows)


Specifying dataset shape:
- `sdf_fhvhv`, raw parquet file, contains 349,585,798 rows x 20 columns
- `sdf_fhvhv_clean`, filtered for rows with valid data, contains 345,254,373 rows x 20 columns 
- `sdf_fhvhv_selected`, filtered for columns to be used downstream, contains 345,254,373 rows x 7 columns

---

## Preprocess MTA Subway Hourly Ridership Data
We want to remove all rows with data anomaly, such as the following: 
- data with `ridership` <= 0
- data with NULL in any of `transit_timestamp`, `station_complex_id`, `station_complex`, `latitude`, `longtitude`, `ridership`

In [25]:
# Read the CSV from directory (all months combined)
sdf_ridership = (
    spark.read
    .option("header", True)
    .csv("../data/raw/mta_ridership/*.csv")
    .select(
        F.to_timestamp("transit_timestamp").alias("transit_timestamp"),
        F.col("station_complex_id").cast("string").alias("station_complex_id"),
        F.col("station_complex").cast("string").alias("station_complex"),
        F.col("borough").cast("string").alias("borough"),
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
        F.col("ridership").cast("double").alias("ridership"),
        F.col("transfers").cast("double").alias("transfers")
    )
)

# Print schema for reference
sdf_ridership.printSchema()

root
 |-- transit_timestamp: timestamp (nullable = true)
 |-- station_complex_id: string (nullable = true)
 |-- station_complex: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ridership: double (nullable = true)
 |-- transfers: double (nullable = true)



In [26]:
# Start with un-filtered data and raw count
n = sdf_ridership.count()
sdf_ridership_clean = sdf_ridership

print(f"Raw rows: {n:,}\n")

# Step 1: required station information
required_station_info = (
    F.col("transit_timestamp").isNotNull() &              # transit_timestamp must not be Null
    F.col("station_complex_id").isNotNull() &             # station_complex_id must not be Null
    F.col("station_complex").isNotNull() &                # station_complex must not be Null
    F.col("latitude").isNotNull() &                       # latitude must not be Null
    F.col("longitude").isNotNull()                        # longitude must not be Null
)
sdf_ridership_clean, n = apply_cleaning_step(sdf_ridership_clean, "Required station information", required_station_info, n)

# Step 2: required time range
required_time = (
    (F.col("transit_timestamp") >= F.lit(START_TIME)) &   # transit time must be within specified time range
    (F.col("transit_timestamp") < F.lit(END_TIME))        # transit time must be within specified time range
)
sdf_ridership_clean, n = apply_cleaning_step(sdf_ridership_clean, "Required time range", required_time, n)

# Step 3: valid ridership fields
valid_ridership = (
    (F.col("ridership") >= 0) &                           # ridership must be non-negative
    (F.col("transfers") >= 0)                             # transfers must be non-negative
)
sdf_ridership_clean, n = apply_cleaning_step(sdf_ridership_clean, "Valid ridership fields", valid_ridership, n)

Raw rows: 5,140,106



Required station information
Before:  5,140,106
After:   5,140,106
Removed: 0 (0.00%)



Required time range
Before:  5,140,106
After:   5,140,106
Removed: 0 (0.00%)



Valid ridership fields
Before:  5,140,106
After:   5,140,101
Removed: 5 (0.00%)



In [5]:
# Remove columns unused in downstream analysis
sdf_ridership_selected = (
    sdf_ridership_clean
    .select(
        F.col("transit_timestamp"),
        F.col("station_complex_id").cast("string"),
        F.col("station_complex").cast("string"),
        F.col("borough").cast("string"),
        F.col("ridership").cast("long"),
        F.col("latitude").cast("double"),
        F.col("longitude").cast("double"),
    )
)
sdf_ridership_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/mta_ridership")

Specifying dataset shape:
- `sdf_ridership`, raw parquet file, contains 5140106 rows x 8 columns
- `sdf_ridership_clean`, filtered for rows with valid data, contains 5140101 rows x 8 columns 
- `sdf_ridership_selected`, filtered for columns to be used downstream, contains 5140101 rows x 7 columns

---

## Preprocess MTA Service Alerts Data
We want to remove all rows with data anomaly, such as the following: 
- alerts with NULL in any of `alert_id`, `event_id`, `date`, `affected`

In [27]:
# Read the CSV from directory (all months combined)
sdf_alerts = (
    spark.read
    .option("header", True).option("multiline", True).option("quote", '"').option("escape", '"')
    .csv("../data/raw/mta_alerts/*.csv")
    .select(
        F.col("alert_id").cast("string").alias("alert_id"),
        F.col("event_id").cast("string").alias("event_id"),
        F.col("update_number").cast("long").alias("update_number"),
        F.to_timestamp("date").alias("date"),
        F.col("agency").cast("string").alias("agency"),
        F.col("status_label").cast("string").alias("status_label"),
        F.col("affected").cast("string").alias("affected"),
        F.col("header").cast("string").alias("header"),
        F.col("description").cast("string").alias("description")
    )
)

# Print schema for reference
sdf_alerts.printSchema()

root
 |-- alert_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- update_number: long (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- status_label: string (nullable = true)
 |-- affected: string (nullable = true)
 |-- header: string (nullable = true)
 |-- description: string (nullable = true)



In [28]:
# Start with un-filtered data and raw count
n = sdf_alerts.count()
sdf_alerts_clean = sdf_alerts

print(f"Raw rows: {n:,}\n")

# Step 1: required alert information
required_alert_info = (
    F.col("alert_id").isNotNull() &                   # alert_id must not be Null
    F.col("event_id").isNotNull() &                   # event_id must not be Null
    F.col("date").isNotNull() &                       # date must not be Null
    F.col("affected").isNotNull()                     # affected must not be Null
)
sdf_alerts_clean, n = apply_cleaning_step(sdf_alerts_clean, "Required alert information", required_alert_info, n)

# Step 2: required agency
required_agency = F.col("agency") == "NYCT Subway"    # agency must be NYCT Subway
sdf_alerts_clean, n = apply_cleaning_step(sdf_alerts_clean, "Required agency", required_agency, n)

# Step 3: required time range
required_time = (
    (F.col("date") >= F.lit(START_TIME)) &            # alert time must be within specified time range
    (F.col("date") < F.lit(END_TIME))                 # alert time must be within specified time range
)
sdf_alerts_clean, n = apply_cleaning_step(sdf_alerts_clean, "Required time range", required_time, n)

Raw rows: 67,502

Required alert information
Before:  67,502
After:   67,502
Removed: 0 (0.00%)

Required agency
Before:  67,502
After:   67,502
Removed: 0 (0.00%)

Required time range
Before:  67,502
After:   67,502
Removed: 0 (0.00%)



In [8]:
# Remove columns unused in downstream analysis
sdf_alerts_selected = (
    sdf_alerts_clean
    .select(
        F.col("event_id"),
        F.col("update_number"),
        F.col("date"),
        F.col("status_label"),
        F.col("affected"),
        F.col("header"),
        F.col("description"),
    )
)

sdf_alerts_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/mta_alerts")

Specifying dataset shape:
- `sdf_alerts`, raw parquet file, contains 67502 rows x 9 columns
- `sdf_alerts_clean`, filtered for rows with valid data, contains 67502 rows x 9 columns 
- `sdf_alerts_selected`, filtered for columns to be used downstream, contains 67502 rows x 7 columns

---

## Preprocess Meteostat Weather Data
We want to remove all rows with data anomaly, such as the following: 
- weather data with NULL in any field that is used in downstream analsyis

In [29]:
# Read the CSV from directory (all months combined)
sdf_weather = (
    spark.read
    .option("header", True)
    .csv("../data/raw/meteostat_weather/*.csv")
    .select(
        F.to_timestamp("time").alias("time"),
        F.col("temp").cast("double").alias("temp"),
        F.col("dwpt").cast("double").alias("dwpt"),
        F.col("rhum").cast("double").alias("rhum"),
        F.col("prcp").cast("double").alias("prcp"),
        F.col("snow").cast("double").alias("snow"),
        F.col("wdir").cast("double").alias("wdir"),
        F.col("wspd").cast("double").alias("wspd"),
        F.col("wpgt").cast("double").alias("wpgt"),
        F.col("pres").cast("double").alias("pres"),
        F.col("tsun").cast("double").alias("tsun"),
        F.col("coco").cast("double").alias("coco")
    )
)

# Print schema for reference
sdf_weather.printSchema()

root
 |-- time: timestamp (nullable = true)
 |-- temp: double (nullable = true)
 |-- dwpt: double (nullable = true)
 |-- rhum: double (nullable = true)
 |-- prcp: double (nullable = true)
 |-- snow: double (nullable = true)
 |-- wdir: double (nullable = true)
 |-- wspd: double (nullable = true)
 |-- wpgt: double (nullable = true)
 |-- pres: double (nullable = true)
 |-- tsun: double (nullable = true)
 |-- coco: double (nullable = true)



In [30]:
# Start with un-filtered data and raw count
n = sdf_weather.count()
sdf_weather_clean = sdf_weather

print(f"Raw rows: {n:,}\n")

# Step 1: required weather information
required_weather_info = (
    F.col("time").isNotNull() &                   # time must not be Null
    F.col("temp").isNotNull() &                   # temp must not be Null
    F.col("rhum").isNotNull() &                   # rhum must not be Null
    F.col("wdir").isNotNull() &                   # wdir must not be Null
    F.col("wspd").isNotNull() &                   # wspd must not be Null
    F.col("coco").isNotNull()                     # coco must not be Null
)
sdf_weather_clean, n = apply_cleaning_step(sdf_weather_clean, "Required weather information", required_weather_info, n)

# Step 2: required time range
required_time = (
    (F.col("time") >= F.lit(START_TIME)) &        # weather entry must be within specified time range
    (F.col("time") < F.lit(END_TIME))             # weather entry must be within specified time range
)
sdf_weather_clean, n = apply_cleaning_step(sdf_weather_clean, "Required time range", required_time, n)

Raw rows: 12,383

Required weather information
Before:  12,383
After:   12,383
Removed: 0 (0.00%)

Required time range
Before:  12,383
After:   12,383
Removed: 0 (0.00%)



In [11]:
# Remove columns unused in downstream analysis
sdf_weather_selected = (
    sdf_weather_clean
    .select(
        F.col("time"),
        F.col("coco").cast("int").alias("coco"),
    )
)

sdf_weather_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/meteostat_weather")

Specifying dataset shape:
- `sdf_weather`, raw parquet file, contains 12383 rows x 12 columns
- `sdf_weather_clean`, filtered for rows with valid data, contains 12383 rows x 12 columns 
- `sdf_weather_selected`, filtered for columns to be used downstream, contains 12383 rows x 2 columns

---

## Checkpoint
By the end of this Jupyter Notebook, all the relevant datasets should have been cleaned (remove anomaly rows, remove unused columns) and saved as a parquet file or directory using PySpark.

For tlc_trips, all rows have a new combined `subtotal` column which describes the amount paid by passenger, excluding any tips (regardless of payment method).

The **processed data structures** are as follows:

tlc_trips / yellow
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `subtotal` = total_amount - tip_amount
- `service_type` = "yellow"

tlc_trips / green
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `subtotal` = total_amount - tip_amount
- `service_type` = "green"

tlc_trips / fhvhv
- `request_datetime`
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `subtotal` = base_passenger_fare + tolls + bcf + sales_tax + congestion_surcharge + airport_fee + cbd_congestion_fee
- `service_type` = "fhvhv"

mta_ridership
- `transit_timestamp`
- `station_complex_id`
- `station_complex`
- `borough`
- `ridership`
- `latitude`
- `longitude`

mta_alerts
- `event_id`
- `update_number`
- `date`
- `status_label`
- `affected`
- `header`
- `description`

meteostat_weather
- `time`
- `coco`

These cleaned datasets will be used for aggregation and feature engineering in the next notebook. 